In [ ]:
import pandas as pd
from sklearn.model_selection import train_test_split
train=pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/train.csv",index_col="Id")
test=pd.read_csv("/kaggle/input/competitions/home-data-for-ml-course/test.csv",index_col="Id")

train.dropna(subset=["SalePrice"],axis=0,inplace=True)
train_y=train.SalePrice

train_X=train.drop(["SalePrice"],axis=1)    

train_X,valid_X,train_y,valid_y=train_test_split(train_X,train_y,random_state=0)

categorial_columns=[col for col in train_X.columns if train_X[col].dtype=='object']
#print(categorial_columns)

#num=[col for col in categorial_columns if train_X[col].nunique()<10]
#print(num)

numerical_columns=[col for col in train_X.columns if train_X[col].dtype in ["int64","float64"]]

#print(numerical_columns)

from sklearn.impute import SimpleImputer
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder

filling_values_category=Pipeline(steps=[
    ('replace_missing',SimpleImputer(strategy='constant',fill_value='Missing')),
    ('encoding',OneHotEncoder(handle_unknown='ignore'))
])

filling_values_numerical=Pipeline(steps=[
    ("filling_nan",SimpleImputer(strategy='median'))
])
preprocessor=ColumnTransformer(transformers=[
    ('categorial',filling_values_category,categorial_columns),
    ('nmerical',filling_values_numerical,numerical_columns)
])

from xgboost import XGBRegressor
model=XGBRegressor(learning_rate=0.05,n_estimators=250,random_state=0)

main_pipeline=Pipeline(steps=[
    ("preprocessing",preprocessor),
    ("model",model)
])

main_pipeline.fit(train_X,train_y)

# from sklearn.metrics import mean_absolute_error

# main_pipeline.fit(train_X, train_y)

# valid_predictions = main_pipeline.predict(valid_X)
# mae = mean_absolute_error(valid_y, valid_predictions)
# print(f"Validation MAE: {mae}")

predictions = main_pipeline.predict(test)
output = pd.DataFrame({
    "Id": test.index,
    "SalePrice": predictions
})
output.to_csv("submission.csv", index=False)
print("Submission saved!")


